# METABRIC Exploratory Data Analysis

**Project:** METABRIC Explorer

This notebook is the first step of the project: understand the dataset before
designing the dashboard. It answers the questions asked in the project brief:

1. What is in the dataset? Is it complete? What do we do with missing data and categorical values?
2. What does the cohort look like, and how are the variables linked?
3. Is there structure in the gene expression data can dimensionality reduction and clustering reveal it?
4. Which questions will a user want to answer, and what does that imply for the dashboard?

**Target user.** A researcher exploring how the molecular subtypes of breast
cancer relate to clinical features, mutations and survival.

**How to run.** Put `METABRIC_RNA_Mutation.csv` in `data/` (see README) and run
all cells from the `notebooks/` folder. Loading and cleaning are done by
`src/data.py`, the same code the dashboard uses, so the notebook and the app
always see the same data.

## 0. Setup

In [1]:
import sys
from pathlib import Path

# make the project's `src` package importable when running from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.cluster import hierarchy
from sklearn.feature_selection import f_classif

from src import analysis, figures
from src.data import (CATEGORICAL_CLINICAL, NUMERIC_CLINICAL, RENAME, category_order,
                      gene_label, label, load_data)

warnings.filterwarnings("ignore", category=FutureWarning)
figures.register_template()
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 30)

data = load_data()
clinical, mrna = data.clinical, data.mrna
mutations, mutation_codes = data.mutations, data.mutation_codes

SUBTYPE_ORDER = category_order(clinical, "pam50_subtype")
SUBTYPES = [s for s in SUBTYPE_ORDER if s != "NC"]          # 'NC' = 6 unclassified patients
SUBTYPE_COLORS = figures.color_map("pam50_subtype", SUBTYPE_ORDER)
subtype = clinical["pam50_subtype"].astype(str)

print(f"{data.n_patients} patients | {len(data.genes)} genes (mRNA) | "
      f"{len(data.mutated_genes)} genes (mutations)")

1904 patients | 489 genes (mRNA) | 173 genes (mutations)


In [2]:
# plot helpers reused throughout the notebook
def embedding_scatter(fig, xy, groups, colors, order, row=1, col=1, showlegend=True, hover=None,
                      legend="legend", prefix=""):
    # one WebGL trace per group so the legend can toggle groups
    for g in order:
        m = (groups == g).to_numpy() if hasattr(groups, "to_numpy") else groups == g
        if m.sum() == 0:
            continue
        fig.add_trace(go.Scattergl(
            x=xy[m, 0], y=xy[m, 1], mode="markers", name=f"{prefix}{g}", legendgroup=f"{prefix}{g}",
            showlegend=showlegend, legend=legend, marker=dict(color=colors[g], size=5, opacity=0.8),
            customdata=np.asarray(clinical.index)[m] if hover is None else hover[m],
            hovertemplate=f"<b>{g}</b><br>patient %{{customdata}}<extra></extra>"),
            row=row, col=col)
    fig.update_xaxes(showticklabels=False, showgrid=False, row=row, col=col)
    fig.update_yaxes(showticklabels=False, showgrid=False, row=row, col=col)


def survival_figure(groups, order, colors, title, event="dss_event", height=430):
    # share of patients alive at each month of the first 5 years, among patients whose status is known
    t, e = clinical["overall_survival_months"], clinical[event]
    fig = go.Figure()
    five_year = []
    for g in order:
        m = (groups == g).to_numpy()
        curve = analysis.survival_share(t[m], e[m])
        fig.add_trace(go.Scatter(
            x=curve["month"], y=curve["survival"], mode="lines",
            name=f"{g} (n={m.sum()})", line=dict(color=colors[g], width=2),
            customdata=curve[["alive", "known"]],
            hovertemplate=(f"<b>{g}</b><br>month %{{x}}: %{{y:.1%}} alive"
                           "<br>(%{customdata[0]} of %{customdata[1]} patients with a known status)<extra></extra>")))
        five_year.append(f"{g} {analysis.survival_at(curve, analysis.HORIZON):.0%}")
    fig.update_layout(
        title=f"{title}<br><sup>Alive at 5 years: {', '.join(five_year)}</sup>",
        height=height, xaxis=dict(title="Months since diagnosis", range=[0, analysis.HORIZON], dtick=12),
        yaxis=dict(title="Patients alive (%)", range=[0, 1.02], tickformat=".0%"))
    return fig


def lower_triangle(matrix):
    # hide the redundant upper triangle and the diagonal of a symmetric matrix
    m = matrix.copy().astype(float)
    m.values[np.triu_indices_from(m, k=0)] = np.nan
    return m.iloc[1:, :-1]

## 1. What is in the dataset?

Each row is one patient. The 693 columns form three blocks of very different
nature, which we handle separately:

In [3]:
raw = pd.read_csv(PROJECT_ROOT / "data" / "METABRIC_RNA_Mutation.csv", low_memory=False)
n_clinical = raw.columns.get_loc("brca1")          # clinical columns come first in the CSV

pd.DataFrame({
    "columns": [n_clinical, len(data.genes), len(data.mutated_genes)],
    "content": ["patient & tumour attributes, treatment, survival",
                "mRNA expression level of 489 genes (z-scores)",
                "mutation code per gene, '0' when the gene is not mutated"],
    "type": ["numeric, ordinal and categorical", "continuous", "text"],
    "missing values": [int(raw.iloc[:, :n_clinical].isna().sum().sum()),
                       int(mrna.isna().sum().sum()), 0],
}, index=["Clinical", "mRNA expression", "Mutations"])

,columns,content,type,missing values
Clinical,31,"patient & tumour attributes, treatment, survival","numeric, ordinal and categorical",1100
mRNA expression,489,mRNA expression level of 489 genes (z-scores),continuous,0
Mutations,173,"mutation code per gene, '0' when the gene is n...",text,0


In [4]:
# Are the mRNA values really z-scores?
per_gene = pd.DataFrame({"mean": mrna.mean(), "std": mrna.std()})
per_gene.agg(["min", "max"]).round(3)

,mean,std
min,-0.0,1.0
max,0.0,1.0


Every gene has mean 0 and standard deviation 1: the expression values are
**already standardised per gene within this cohort**. No extra scaling is
needed before dimensionality reduction or clustering (scaling again would
change nothing).

> Note: the project brief mentions 331 genes and 175 mutation columns; the
> actual file has **489 genes and 173 mutation columns**.

## 2. Data quality: missing values and categorical values

### 2.1 Missing values

Only the clinical block has missing values.

In [5]:
missing = raw.iloc[:, :n_clinical].isna().mean().mul(100)
missing = missing[missing > 0].sort_values()
names = [label(RENAME.get(c, c)) for c in missing.index]

fig = go.Figure(go.Bar(
    x=missing.values, y=names, orientation="h", marker_color=figures.CATEGORICAL[0],
    text=[f"{v:.1f}%" for v in missing.values], textposition="outside", cliponaxis=False,
    hovertemplate="%{y}: %{x:.1f}% of patients<extra></extra>"))
fig.update_layout(title="Missing values per clinical column (only columns with missing values)",
                  xaxis_title="Missing (% of patients)", height=420, margin=dict(r=48))
fig.update_xaxes(range=[0, 30])
fig

**What we do with them** (implemented in `src/data.py`):

| Column | Missing | Decision | Why |
|---|---|---|---|
| Tumour stage | 26% | keep, as an explicit **"Unknown"** category | clinically important; dropping the column (as in our first draft) or the rows (26% of patients) would lose too much |
| 3-gene classifier, grade, cellularity, laterality, surgery, ER (IHC), histology | 1–11% | **"Unknown"** category | the user sees how much is missing instead of a guessed value |
| Tumour size | 1% | keep as NaN, left out per chart | imputing a size would invent data |
| Mutation count | 2.4% | **set to 0** | the 45 missing values are exactly the 45 patients with no mutated gene in the panel: missing means "no mutation found" |
| Death from cancer | 1 patient | "Unknown" cause | deceased, cause not recorded |

We deliberately **do not impute** clinical values: the dashboard is for
exploration, and an imputed value would be shown as if it were real. The
algorithms (DR and clustering) run on the mRNA block, which is complete.

### 2.2 Categorical values: the mutation columns

The 173 mutation columns are text. They contain the exact mutation(s), e.g.
for *PIK3CA*:

In [6]:
codes = mutation_codes["pik3ca"]
codes[codes != ""].value_counts().head(6).rename("patients").to_frame()

,patients
pik3ca,
H1047R,285
E545K,127
E542K,72
N345K,42
H1047L,29
C420R,12


In [7]:
n_dummies = sum(mutation_codes[g][mutation_codes[g] != ""].nunique() for g in data.mutated_genes)
print(f"One-hot encoding the raw codes  -> {n_dummies:,} columns, most used by 1 or 2 patients")
print(f"Binary flag 'gene is mutated'   -> {len(data.mutated_genes)} columns")

One-hot encoding the raw codes  -> 7,757 columns, most used by 1 or 2 patients
Binary flag 'gene is mutated'   -> 173 columns


Our first draft one-hot encoded these codes, producing a 1,904 × 8,494 matrix.
We now keep one **binary flag per gene** (mutated / not mutated): it is the
level at which a biologist reasons ("is *TP53* mutated?"), and it keeps
mutation frequencies comparable across genes. The raw codes are kept for the
dashboard tooltips.

Other categorical variables (subtype, receptor status, grade, …) are kept as
labelled categories with a fixed display order, so every chart and filter
lists them the same way. Ordinal ones (grade, stage, cellularity) keep their
natural order.

### 2.3 Distribution of the mRNA z-scores

In [8]:
values = mrna.to_numpy().ravel()
counts, edges = np.histogram(values, bins=np.arange(-8, 21, 0.25))
centres = (edges[:-1] + edges[1:]) / 2

fig = go.Figure(go.Bar(x=centres, y=counts, width=0.25, marker_color=figures.CATEGORICAL[0],
                       hovertemplate="z ≈ %{x:.2f}: %{y:,} values<extra></extra>"))
for x in figures.ZSCORE_RANGE:
    fig.add_vline(x=x, line_dash="dash", line_color=figures.TEXT_SECONDARY)
fig.add_annotation(x=3, y=np.log10(counts.max()), text="  colour scale clipped at ±3",
                   showarrow=False, xanchor="left", font_color=figures.TEXT_SECONDARY)
fig.update_layout(title="All mRNA z-scores (489 genes × 1,904 patients)", height=360,
                  xaxis_title="mRNA z-score",
                  yaxis=dict(type="log", dtick=1, title="Number of values (log scale)"))
fig

In [9]:
print(f"Values beyond ±3: {(np.abs(values) > 3).mean():.2%}   |   maximum: {values.max():.1f}")

Values beyond ±3: 0.93%   |   maximum: 20.4


The distribution is centred on 0 but skewed to the right: some genes are
strongly over-expressed in a few patients (up to z = 20), while
under-expression stops around z = −7. In the
dashboard, colour scales for expression are **clipped at ±3** so that these
rare extremes do not wash out the colours of the other 99% of the values; the
exact value stays available in the tooltip.

## 3. The cohort at a glance

### 3.1 Molecular subtypes

PAM50 is a gene-expression classifier that sorts breast tumours into intrinsic
subtypes (Luminal A, Luminal B, HER2-enriched, Basal-like, Claudin-low,
Normal-like). It is the main label we use to colour the data; its colours are
fixed for the whole project (colour-blind-safe Okabe–Ito palette).

In [10]:
counts = clinical["pam50_subtype"].value_counts().reindex(SUBTYPE_ORDER)
fig = go.Figure(go.Bar(x=counts.index, y=counts.values, text=counts.values, textposition="outside",
                       marker_color=[SUBTYPE_COLORS[s] for s in counts.index], cliponaxis=False,
                       hovertemplate="%{x}: %{y} patients<extra></extra>"))
fig.update_layout(title="Patients per PAM50 subtype ('NC' = not classified)",
                  xaxis_title="PAM50 subtype", yaxis_title="Patients", height=360)
fig

### 3.2 Clinical measurements (dashed line = median)

In [11]:
cols = ["age_at_diagnosis", "tumor_size", "lymph_nodes_examined_positive", "nottingham_prognostic_index"]
fig = make_subplots(rows=1, cols=4, subplot_titles=[label(c) for c in cols], horizontal_spacing=0.06)
for i, col in enumerate(cols, start=1):
    v = clinical[col].dropna()
    fig.add_trace(go.Histogram(x=v, nbinsx=30, marker_color=figures.CATEGORICAL[0], showlegend=False,
                               hovertemplate="%{x}: %{y} patients<extra></extra>"), 1, i)
    fig.add_vline(x=v.median(), line_dash="dash", line_color=figures.TEXT, row=1, col=i)
fig.update_yaxes(title_text="Patients", row=1, col=1)
fig.update_annotations(font_size=12)
fig.update_layout(title="Distribution of the clinical measurements (dashed line = median)",
                  height=340, margin=dict(t=80))
fig

In [12]:
clinical[cols].describe().T[["count", "mean", "50%", "min", "max"]].rename(index=label).round(1)

,count,mean,50%,min,max
Age at diagnosis (years),1904.0,61.1,61.8,21.9,96.3
Tumor size (mm),1884.0,26.2,23.0,1.0,182.0
Positive lymph nodes (count),1904.0,2.0,0.0,0.0,45.0
Nottingham Prognostic Index,1904.0,4.0,4.0,1.0,6.4


- **Age** is roughly normal around 61 years.
- **Tumour size** and especially **positive lymph nodes** are strongly right-skewed:
  most patients have 0 positive nodes, a few have more than 20. In the dashboard
  these variables need a log or binned scale option, otherwise the bulk of the
  patients is squeezed into one bar.
- The **Nottingham Prognostic Index** combines size, nodes and grade, so we
  expect it to correlate with them (section 4).

### 3.3 Hormone receptors

In [13]:
rows = []
for col in ["er_status", "pr_status", "her2_status"]:
    for status, share in clinical[col].value_counts(normalize=True).items():
        rows.append({"receptor": label(col), "status": status, "share": 100 * share})
fig = px.bar(pd.DataFrame(rows), x="share", y="receptor", color="status", orientation="h",
             text=pd.DataFrame(rows)["share"].map("{:.0f}%".format),
             color_discrete_map={"Positive": figures.CATEGORICAL[0], "Negative": figures.OTHER_COLOR},
             category_orders={"status": ["Positive", "Negative"]})
fig.update_traces(hovertemplate="%{y} %{fullData.name}: %{x:.1f}%<extra></extra>")
fig.update_yaxes(categoryorder="array", categoryarray=["HER2 status", "PR status", "ER status"])
fig.update_layout(title="Receptor status", xaxis_title="Patients (%)", yaxis_title=None,
                  legend_title_text="Status", height=260)
fig

## 4. How are the variables linked?

### 4.1 Numeric clinical variables, Spearman correlation

Spearman (rank) correlation is used rather than Pearson because tumour size and
lymph nodes are very skewed. Diverging colour scale centred on 0; only the
lower triangle is shown since the matrix is symmetric and the diagonal is
always 1.

In [14]:
num = ["age_at_diagnosis", "tumor_size", "lymph_nodes_examined_positive",
       "nottingham_prognostic_index", "mutation_count", "overall_survival_months"]
corr = lower_triangle(clinical[num].corr(method="spearman").rename(index=label, columns=label))
fig = px.imshow(corr, text_auto=".2f", zmin=-1, zmax=1, color_continuous_scale=figures.DIVERGING,
                aspect="auto")
fig.update_traces(hovertemplate="%{y} × %{x}<br>ρ = %{z:.2f}<extra></extra>")
fig.update_layout(title="Spearman correlation between clinical measurements", height=460,
                  coloraxis_colorbar_title="ρ", xaxis_showgrid=False, yaxis_showgrid=False)
fig

The Nottingham Prognostic Index is strongly linked to positive lymph nodes
(ρ ≈ 0.73) and tumour size (ρ ≈ 0.52) — as expected, since it is computed from
them. The other measurements are nearly independent of each other. Follow-up
time is negatively correlated with NPI, size and nodes (worse tumours, shorter
survival).

### 4.2 Categorical variables, mutual information

Mutual information (practical session 1) measures how much knowing one
variable tells us about another. We use its **normalized** version (divided by
the entropies of the two variables) so that every pair lies between 0
(independent) and 1 (one determines the other), whatever their number of
categories. Sequential colour scale since the values are non-negative; the
diagonal (always 1) is hidden so that it does not dominate the scale.

In [15]:
cat = ["pam50_subtype", "er_status", "pr_status", "her2_status", "three_gene_subtype",
       "integrative_cluster", "neoplasm_histologic_grade", "tumor_stage", "cellularity",
       "inferred_menopausal_state", "chemotherapy", "hormone_therapy", "vital_status"]
MI = lower_triangle(analysis.mutual_information_matrix(clinical, cat).rename(index=label, columns=label))
fig = px.imshow(MI, text_auto=".2f", zmin=0, color_continuous_scale=figures.SEQUENTIAL, aspect="auto")
fig.update_traces(hovertemplate="%{y} × %{x}<br>normalized MI = %{z:.2f}<extra></extra>")
fig.update_layout(title="Association between categorical variables (normalized mutual information)", height=620,
                  coloraxis_colorbar_title="NMI", xaxis_showgrid=False, yaxis_showgrid=False)
fig

The molecular classifications share the most information with each other and
with the receptors: PAM50 <-> 3-gene classifier (0.32), PAM50 <-> ER status and
ER <-> 3-gene classifier (0.29), ER <-> PR (0.27). Chemotherapy is mostly linked
to ER status (0.18: ER-negative tumours get chemotherapy). Vital status shares
almost no information with any single variable (at most 0.05), which already
suggests that survival is hard to predict from one feature alone. Normalized MI
values are naturally lower than correlations: 0.3 is already a strong
association.

### 4.3 Two molecular classifications compared

In [16]:
ic_order = category_order(clinical, "integrative_cluster")
table = pd.crosstab(clinical["pam50_subtype"], clinical["integrative_cluster"], normalize="index")
table = table.reindex(index=SUBTYPES, columns=ic_order).mul(100)
fig = px.imshow(table, text_auto=".0f", color_continuous_scale=figures.SEQUENTIAL, zmin=0, aspect="auto",
                labels=dict(x="Integrative cluster", y="PAM50 subtype", color="% of subtype"))
fig.update_traces(hovertemplate="%{y} → cluster %{x}: %{z:.0f}% of the subtype<extra></extra>")
fig.update_layout(title="Where does each PAM50 subtype fall among the integrative clusters? (row %)",
                  height=380, xaxis_showgrid=False, yaxis_showgrid=False)
fig

The two classifications agree only partly: Basal tumours concentrate in
integrative cluster 10 and Her2 tumours in cluster 5, while the luminal
subtypes are spread over several clusters. Such a cross-tabulation, with the
ability to click a cell to select those patients, is a natural linked view for
the dashboard.

### 4.4 Age by subtype

In [17]:
fig = px.box(clinical[subtype.isin(SUBTYPES)].assign(pam50_subtype=subtype), x="pam50_subtype",
             y="age_at_diagnosis", color="pam50_subtype", color_discrete_map=SUBTYPE_COLORS,
             category_orders={"pam50_subtype": SUBTYPES}, points=False,
             labels={"pam50_subtype": label("pam50_subtype"), "age_at_diagnosis": label("age_at_diagnosis")})
fig.update_layout(title="Age at diagnosis by PAM50 subtype", showlegend=False, height=380)
fig

Basal-like and Her2 patients are diagnosed younger than Luminal A patients.
A box plot was chosen over a violin plot here because the distributions are
unimodal; the median and quartiles are the information that matters.

## 5. Survival

`overall_survival_months` is a follow-up time. A patient who is alive at the
end of follow-up only tells us she survived *at least* that long (her time is
**censored**), so the share of deaths alone would be misleading: patients
followed for 2 years and for 20 years would count the same.

We use a simple, course-level measure: for each month t after diagnosis, the
**share of patients still alive among the patients whose status at t is
known** (alive = followed for more than t months; dead = died at or before t;
patients whose follow-up ended before t, still alive, are left out). It is a
proportion per month drawn as a line chart (`analysis.survival_share`).

We show it over the **first 5 years** only. Patients were recruited over many
years, so at long horizons the patients recruited later (still alive, but not
yet followed that long) drop out of the share while early deaths stay in, and
the share falls too fast. Up to 5 years almost every patient's status is known
and the share agrees with the standard Kaplan-Meier estimate within one point.

We use **disease-specific survival** (death from breast cancer = event; death
from another cause is not counted as an event), because 480 patients died of
other causes, which says little about their tumour.

In [18]:
survival_figure(subtype, SUBTYPES, SUBTYPE_COLORS, "Disease-specific survival over 5 years by PAM50 subtype")

In [19]:
rows = []
for s in SUBTYPES:
    m = (subtype == s).to_numpy()
    curve = analysis.survival_share(clinical["overall_survival_months"][m], clinical["dss_event"][m])
    rows.append({"subtype": s, "patients": int(m.sum()),
                 "alive at 2 years": analysis.survival_at(curve, 24),
                 "alive at 5 years": analysis.survival_at(curve, 60)})
pd.DataFrame(rows).set_index("subtype").style.format({"alive at 2 years": "{:.0%}", "alive at 5 years": "{:.0%}"})

,patients,alive at 2 years,alive at 5 years
subtype,,,
LumA,679,98%,93%
LumB,461,98%,80%
Her2,220,91%,67%
Basal,199,84%,63%
claudin-low,199,91%,82%
Normal,140,96%,84%


Survival differs strongly between subtypes. Luminal A has the best prognosis
(93% alive at 5 years). Basal-like and Her2 tumours have the worst survival in
the first 5 years (63% and 67%): most of their deaths happen early. This is the
kind of pattern a survival curve shows and a bar chart of "% deceased" would
hide.

## 6. Gene expression and mutations

### 6.1 Sanity check: do the genes agree with the clinical labels?

The panel does not contain the classic ER marker *ESR1*, but it contains
*GATA3*, a transcription factor of ER-positive cells, and *ERBB2*, the gene
that encodes HER2.

In [20]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["GATA3 by ER status", "ERBB2 by HER2 status"])
for i, (gene, col) in enumerate([("gata3", "er_status"), ("erbb2", "her2_status")], start=1):
    for j, status in enumerate(["Positive", "Negative"]):
        m = (clinical[col] == status).to_numpy()
        fig.add_trace(go.Box(y=mrna[gene][m], name=status, boxpoints=False, showlegend=False,
                             marker_color=[figures.CATEGORICAL[0], figures.OTHER_COLOR][j]), 1, i)
    fig.update_yaxes(title_text=gene_label(gene), row=1, col=i)
fig.update_annotations(font_size=12)
fig.update_layout(title="Marker genes versus clinical receptor status", height=380, margin=dict(t=80))
fig

Clear separation in both cases: the expression data is consistent with the
clinical labels, which increases our confidence in the data.

### 6.2 Which genes differ most between subtypes?

Mean z-score per subtype for the 25 genes whose expression differs most
between subtypes (one-way ANOVA F-statistic). Diverging scale: red = higher
than average, blue = lower.

In [21]:
m = subtype.isin(SUBTYPES).to_numpy()
F, _ = f_classif(mrna[m], subtype[m])
top_genes = pd.Series(F, index=data.genes).nlargest(25).index
means = mrna.loc[m, top_genes].groupby(subtype[m]).mean().reindex(SUBTYPES).T
lim = float(np.abs(means.values).max())
fig = px.imshow(means.rename(index=str.upper), color_continuous_scale=figures.DIVERGING,
                zmin=-lim, zmax=lim, aspect="auto",
                labels=dict(x="PAM50 subtype", y="Gene", color="mean z-score"))
fig.update_traces(hovertemplate="%{y} in %{x}: mean z = %{z:.2f}<extra></extra>")
fig.update_layout(title="Mean expression of the 25 most subtype-specific genes", height=620,
                  xaxis_showgrid=False, yaxis_showgrid=False)
fig

Basal-like tumours have the most distinct profile: low *GATA3*, *BCL2*,
*MAPT*, *ERBB3* (genes of the luminal / ER programme) and high cell-cycle
genes (*CCNE1*, *AURKA*, *CDK1*, *E2F2/3*), they are highly proliferative.
Luminal A shows the opposite pattern, and what separates Luminal B from
Luminal A is mainly higher proliferation (*AURKA*, *CDK1*), which matches the
clinical definition of these subtypes.

### 6.3 Mutations by subtype

In [22]:
top_mut = mutations.mean().nlargest(15).index
freq = mutations.loc[m, top_mut].groupby(subtype[m]).mean().reindex(SUBTYPES).T.mul(100)
freq.insert(0, "All", mutations[top_mut].mean().mul(100))
fig = px.imshow(freq.rename(index=str.upper), text_auto=".0f", color_continuous_scale=figures.SEQUENTIAL,
                zmin=0, aspect="auto", labels=dict(x="PAM50 subtype", y="Gene", color="% mutated"))
fig.update_traces(hovertemplate="%{y} mutated in %{z:.0f}% of %{x} patients<extra></extra>",
                  xgap=0, ygap=0)
fig.add_vline(x=0.5, line_color=figures.SURFACE, line_width=6)   # sets the "All" column apart
fig.update_layout(title="Share of patients with a mutation (15 most frequently mutated genes)",
                  height=560, xaxis_showgrid=False, yaxis_showgrid=False)
fig

Two genes dominate: *PIK3CA* (42% of all patients, 57% of Luminal A) and
*TP53* (35% overall, but 88% of Basal-like and 70% of Her2 tumours).
*GATA3* mutations are mostly found in luminal tumours and are absent from
Basal-like ones. Mutation status is
therefore strongly subtype-dependent: the dashboard should let the user
compare mutation frequencies between any selected group and the rest.

## 7. Dimensionality reduction

All methods are applied to the **489 mRNA z-scores** (complete, already on the
same scale). Clinical variables are used to *colour* the result, not to compute
it, so we can see whether the gene expression structure matches clinical labels.

### 7.1 PCA (linear)

In [23]:
X_raw = mrna.to_numpy()
P_raw, pca_raw = analysis.pca(X_raw)
ratio = pca_raw.explained_variance_ratio_
cumulative = np.cumsum(ratio)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Scree plot (first 30 components)",
                                                    "Cumulative explained variance"])
fig.add_trace(go.Bar(x=np.arange(1, 31), y=ratio[:30], marker_color=figures.CATEGORICAL[0],
                     hovertemplate="PC%{x}: %{y:.1%}<extra></extra>", showlegend=False), 1, 1)
fig.add_trace(go.Scatter(x=np.arange(1, len(ratio) + 1), y=cumulative, mode="lines",
                         line=dict(color=figures.CATEGORICAL[0], width=2), showlegend=False,
                         hovertemplate="%{x} components: %{y:.1%}<extra></extra>"), 1, 2)
for level in (0.5, 0.8):
    k = int(np.searchsorted(cumulative, level) + 1)
    fig.add_hline(y=level, line_dash="dot", line_color=figures.TEXT_MUTED, row=1, col=2)
    fig.add_annotation(x=k, y=level, text=f"{level:.0%}: {k} PCs", showarrow=True, arrowhead=0,
                       ax=40, ay=-25, row=1, col=2)
fig.update_xaxes(title_text="Principal component", row=1, col=1)
fig.update_xaxes(title_text="Number of components", row=1, col=2)
fig.update_yaxes(title_text="Explained variance", tickformat=".0%", row=1, col=1)
fig.update_yaxes(tickformat=".0%", range=[0, 1.02], row=1, col=2)
fig.update_annotations(font_size=12)
fig.update_layout(title="How much variance does each principal component explain? (raw data)",
                  height=380, margin=dict(t=80))
fig

The variance is spread over many components: PC1 explains only ~8%, and 33
components are needed for half of the variance. There is no sharp "elbow", so
a 2-D PCA view will necessarily show only a small part of the structure.

### 7.2 What does the first component capture? A batch effect.

Colouring PC1/PC2 by subtype and by **sampling cohort** (the batch in which the
tumours were processed):

In [24]:
cohort = clinical["cohort"].astype(int).astype(str)
cohort_order = sorted(cohort.unique())
cohort_colors = figures.color_map("cohort", cohort_order)


def pca_subtype_vs_cohort(P, title):
    # same PC1/PC2 projection twice: coloured by subtype, then by cohort (two legends)
    fig = make_subplots(rows=1, cols=2, subplot_titles=["Coloured by PAM50 subtype",
                                                        "Coloured by sampling cohort"],
                        horizontal_spacing=0.04)
    embedding_scatter(fig, P[:, :2], subtype, SUBTYPE_COLORS, SUBTYPE_ORDER, 1, 1)
    embedding_scatter(fig, P[:, :2], cohort, cohort_colors, cohort_order, 1, 2,
                      legend="legend2", prefix="cohort ")
    for c in (1, 2):
        fig.update_xaxes(title_text="PC1", row=1, col=c)
    fig.update_yaxes(title_text="PC2", row=1, col=1)
    fig.update_annotations(font_size=12)
    fig.update_layout(title=title, height=480,
                      legend=dict(title_text="PAM50 subtype", y=1, yanchor="top"),
                      legend2=dict(title_text="Sampling cohort", y=0.42, yanchor="top",
                                   x=1.02, xanchor="left", itemsizing="constant",
                                   font=dict(size=figures.FONT_SIZE - 1, color=figures.TEXT_SECONDARY),
                                   title_font=dict(size=figures.FONT_SIZE - 1, color=figures.TEXT_SECONDARY)))
    return fig


pca_subtype_vs_cohort(P_raw, "PCA of the raw mRNA z-scores")

In [25]:
def pc_associations(P, n=4):
    # normalized mutual information between each PC (cut in quartiles) and a few labels
    out = {}
    for col in ["cohort", "pam50_subtype", "er_status", "neoplasm_histologic_grade"]:
        out[label(col)] = [analysis.mutual_information(pd.qcut(P[:, i], 4, labels=False), clinical[col]) for i in range(n)]
    return pd.DataFrame(out, index=[f"PC{i + 1}" for i in range(n)]).T.round(2)

print("Normalized mutual information between each principal component and the labels (raw data)")
pc_associations(P_raw)

Normalized mutual information between each principal component and the labels (raw data)


,PC1,PC2,PC3,PC4
Cohort,0.17,0.00,0.01,0.05
Molecular subtype (PAM50 + claudin-low),0.04,0.27,0.14,0.01
ER status,0.01,0.29,0.03,0.00
Histologic grade,0.01,0.16,0.00,0.00


**PC1 is mostly the sampling cohort** (cohorts 1-2 on one side, cohort 3 on the
other), not biology. The biological signal (ER status, subtype, grade) is in
PC2. This is a classic **batch effect**: tumours processed in different batches
have systematic technical differences. If we ignore it, clustering will partly
find the batches instead of tumour types.

### 7.3 Removing the cohort effect

The cohorts contain almost the same mix of subtypes (normalized mutual information cohort <-> PAM50
below), so each cohort can be centred on its own mean, gene by gene, without
removing real biological differences (function `remove_cohort_effect` below).

In [26]:
print(f"Normalized mutual information cohort vs PAM50 subtype: {analysis.mutual_information(cohort, subtype):.3f} "
      "(close to 0: same subtype mix in every cohort)")
def remove_cohort_effect(expr, groups):
    # centre every gene within each cohort, then re-standardise (mean 0, sd 1)
    centred = expr - expr.groupby(groups).transform("mean")
    return (centred - centred.mean()) / centred.std()


mrna_c = remove_cohort_effect(mrna, clinical["cohort"])
X = mrna_c.to_numpy()                      # corrected matrix used from now on
P, pca_c = analysis.pca(X)
print("Normalized mutual information between each principal component and the labels (after correction)")
pc_associations(P)

Normalized mutual information cohort vs PAM50 subtype: 0.015 (close to 0: same subtype mix in every cohort)


Normalized mutual information between each principal component and the labels (after correction)


,PC1,PC2,PC3,PC4
Cohort,0.00,0.00,0.01,0.01
Molecular subtype (PAM50 + claudin-low),0.27,0.11,0.07,0.01
ER status,0.28,0.03,0.01,0.00
Histologic grade,0.16,0.00,0.00,0.00


In [27]:
pca_subtype_vs_cohort(P, "PCA after removing the cohort effect")

After correction the cohorts are mixed and PC1 now follows ER status and
subtype: Basal-like tumours on one side, luminal tumours on the other, with
Her2 and claudin-low in between. **From here on we use the corrected matrix.**
The dashboard shows the expression data as published: the correction was judged
too hard to interpret for a dashboard user, so it is only explored in this notebook.

### 7.4 Non-linear methods: t-SNE and UMAP

In [28]:
T = analysis.tsne(X, perplexity=30)
U = analysis.umap(X, n_neighbors=15, min_dist=0.1)

fig = make_subplots(rows=1, cols=3, subplot_titles=["PCA (PC1 × PC2)", "t-SNE (perplexity 30)",
                                                    "UMAP (15 neighbours, min_dist 0.1)"],
                    horizontal_spacing=0.03)
for i, Y in enumerate([P[:, :2], T, U], start=1):
    embedding_scatter(fig, Y, subtype, SUBTYPE_COLORS, SUBTYPE_ORDER, 1, i, showlegend=(i == 1))
fig.update_annotations(font_size=12)
fig.update_layout(title="Three 2-D views of the same 489-dimensional data, coloured by PAM50 subtype",
                  height=440)
fig

In all three views the patients form **one continuous cloud, not separate
islands**, with a gradient from luminal (blue) to Basal-like (vermillion)
tumours. Basal-like is the most distinct group; Luminal A / Luminal B / Normal
overlap strongly. The only real island, visible in t-SNE and UMAP, is a small
group of 18 patients examined in section 8.2. Note that the axes of t-SNE and
UMAP have no meaning (only neighbourhoods do), so their tick labels are hidden.

Why is the separation weak? The 489 genes of this panel are cancer-related
genes, not the 50 genes used to define PAM50, so the subtype signal is diluted
among many genes. The subtype information is nevertheless present: a
15-nearest-neighbour classifier recovers the subtype far better than chance:

In [29]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

m = subtype.isin(SUBTYPES).to_numpy()
baseline = subtype[m].value_counts(normalize=True).max()
for name, Y in [("489 genes", X), ("PCA, 10 components", P[:, :10]), ("t-SNE 2-D", T), ("UMAP 2-D", U)]:
    acc = cross_val_score(KNeighborsClassifier(15), Y[m], subtype[m], cv=5).mean()
    print(f"{name:20s} subtype recovered for {acc:.0%} of patients  (always guessing LumA: {baseline:.0%})")

489 genes            subtype recovered for 63% of patients  (always guessing LumA: 36%)
PCA, 10 components   subtype recovered for 69% of patients  (always guessing LumA: 36%)
t-SNE 2-D            subtype recovered for 63% of patients  (always guessing LumA: 36%)


UMAP 2-D             subtype recovered for 64% of patients  (always guessing LumA: 36%)


### 7.5 Which projection is most faithful? (R<sub>NX</sub> quality curve)

A 2-D projection always distorts the data. Following the practical session on
dimensionality reduction, we measure **neighbourhood preservation**: for each
neighbourhood size *K*, R<sub>NX</sub>(K) tells how many of a patient's *K*
nearest neighbours in the 489-D space remain among its *K* nearest neighbours in
the 2-D view (0 = no better than random, 1 = perfect). The area under the curve
(AUC, log-scale in *K*) summarises it in one number (Lee & Verleysen, 2009).

In [30]:
fig = go.Figure()
for i, (name, Y) in enumerate([("PCA", P[:, :2]), ("t-SNE", T), ("UMAP", U)]):
    curve = analysis.rnx_curve(X, Y)
    fig.add_trace(go.Scatter(x=curve["K"], y=curve["R_NX"], mode="lines", line=dict(width=2, color=figures.CATEGORICAL[i]),
                             name=f"{name} (AUC = {analysis.rnx_auc(curve):.3f})",
                             hovertemplate=f"{name}<br>K = %{{x}}: R_NX = %{{y:.3f}}<extra></extra>"))
fig.update_layout(title="Neighbourhood preservation of the 2-D projections", height=400,
                  xaxis=dict(type="log", dtick=1, title="Neighbourhood size K (log scale)"),
                  yaxis=dict(title="R<sub>NX</sub>(K)", rangemode="tozero"), legend_title_text="Method")
fig

t-SNE preserves small neighbourhoods best (it is designed for local structure),
UMAP is in between, and PCA is the weakest locally but becomes the best for
large neighbourhoods (K > ~300: global structure). There is no single "best"
projection: it depends on whether the user looks at close neighbours or at the
overall layout. All values are low in absolute terms, which
confirms that 2 dimensions cannot hold the structure of this data: the
dashboard must **show the quality curve next to the projection**, so the user
does not over-interpret visual clusters.

## 8. Clustering

Clustering is run on the corrected 489-D expression matrix (not on the 2-D
projections), and the result is compared with the PAM50 subtypes using
**purity**: the share of patients whose cluster's most frequent subtype is
their own subtype (1 = every cluster holds a single subtype). Purity grows with
the number of clusters, so we always compare it at the same k and with the
baseline of putting everybody in one cluster (the share of Luminal A, 0.36).

### 8.1 K-means: how many clusters?

The three criteria from the practical session and lecture 06: within-cluster
sum of squares *W*, between-cluster sum of squares *B* and the
Calinski–Harabasz index, for k = 2 … 10:

In [31]:
choose_k = analysis.choose_k_table(X, range(2, 11))
panels = [("within_ss", "Within-cluster SS (W) ↓"), ("between_ss", "Between-cluster SS (B) ↑"),
          ("calinski_harabasz", "Calinski–Harabasz index ↑")]
fig = make_subplots(rows=1, cols=3, subplot_titles=[t for _, t in panels], horizontal_spacing=0.08)
for i, (col, _) in enumerate(panels, start=1):
    fig.add_trace(go.Scatter(x=choose_k["k"], y=choose_k[col], mode="lines+markers", showlegend=False,
                             line=dict(color=figures.CATEGORICAL[0], width=2), marker=dict(size=8),
                             hovertemplate="k = %{x}: %{y:,.3f}<extra></extra>"), 1, i)
    fig.update_xaxes(title_text="k", dtick=1, row=1, col=i)
fig.update_yaxes(rangemode="tozero")       # y-axes start at 0: no exaggerated drops
fig.update_annotations(font_size=12)
fig.update_layout(title="Choosing the number of K-means clusters", height=350, margin=dict(t=80))
fig

There is **no clear elbow** in *W*: going from 2 to 10 clusters reduces the
within-cluster variance by only ~13%. *B* grows smoothly, and the
Calinski–Harabasz index (their ratio, corrected for k) is highest at k = 2 and
then decreases steadily: the data has no well-separated clusters, consistent
with the continuous cloud seen in section 7. Lecture 06 calls choosing k an
open problem, using precisely the example of "2 versus 3 breast cancer
subtypes". The best-scoring split, k = 2, is still
meaningful:

In [32]:
km2 = pd.Series(analysis.kmeans(X, 2), index=clinical.index).map({0: "Cluster A", 1: "Cluster B"})
print(f"K-means k=2 vs ER status: normalized mutual information = "
      f"{analysis.mutual_information(km2, clinical['er_status']):.2f}")
pd.crosstab(km2, clinical["pam50_subtype"]).reindex(columns=SUBTYPE_ORDER)

K-means k=2 vs ER status: normalized mutual information = 0.52


pam50_subtype,LumA,LumB,Her2,Basal,claudin-low,Normal,NC
row_0,,,,,,,
Cluster A,4,43,155,193,133,15,0
Cluster B,675,418,65,6,66,125,6


The two K-means clusters reproduce the main biological division of breast
cancer: **ER-positive / luminal** versus **ER-negative / Basal-like + Her2**
tumours. With k = 6 (the number of PAM50 subtypes):

In [33]:
k = 6
km = pd.Series(analysis.kmeans(X, k), index=clinical.index).map(lambda c: f"K{c + 1}")
km_order = [f"K{i + 1}" for i in range(k)]
baseline_purity = subtype.value_counts(normalize=True).max()
print(f"Purity of K-means (k = {k}) w.r.t. PAM50: {analysis.purity(km, subtype):.2f} "
      f"(one cluster for everybody: {baseline_purity:.2f})")
share = pd.crosstab(km, clinical["pam50_subtype"], normalize="index").reindex(index=km_order, columns=SUBTYPES).mul(100)
fig = px.imshow(share, text_auto=".0f", color_continuous_scale=figures.SEQUENTIAL, zmin=0, zmax=100, aspect="auto",
                labels=dict(x="PAM50 subtype", y="K-means cluster", color="% of cluster"))
fig.update_traces(hovertemplate="%{y}: %{z:.0f}% %{x}<extra></extra>")
fig.update_layout(title=f"Composition of the K-means clusters (k = {k}), row %", height=360,
                  xaxis_showgrid=False, yaxis_showgrid=False)
fig

Purity of K-means (k = 6) w.r.t. PAM50: 0.59 (one cluster for everybody: 0.36)


Three clusters are each dominated by one subtype (Basal-like together with
claudin-low, Luminal B, Luminal A); the others are mixtures. Purity is well
above the one-cluster baseline but far from 1, as expected from a continuum. **What are the
clusters useful for?** One answer: do they separate patients with different
prognosis?

In [34]:
survival_figure(km, km_order, figures.color_map("kmeans", km_order),
          f"Disease-specific survival by K-means cluster (k = {k})")

Yes: the clusters, found from gene expression alone without any clinical
information, have clearly different survival over the first 5 years (values
in the subtitle). Linking the clustering view to a survival view will be a key
feature of the dashboard.

### 8.2 Hierarchical clustering

Agglomerative clustering builds a tree (dendrogram) by repeatedly merging the
two closest clusters; how "closest" is defined is the **linkage**. We compare
the linkages of lectures 05 and 06 (single, complete, average and **centroid**,
the distance between cluster means, described as "the standard in biology")
with **Ward** (merge the pair whose fusion increases the within-cluster
variation *W* least, i.e. the K-means criterion of lecture 04 applied greedily):

In [35]:
rows = {}
for linkage in ["ward", "centroid", "average", "complete", "single"]:
    Z_ = analysis.linkage_tree(X, linkage)
    labels_ = analysis.cut_tree(Z_, 5)
    rows[linkage] = {"cluster sizes (k = 5)": sorted(np.bincount(labels_).tolist(), reverse=True),
                     "purity vs PAM50": round(analysis.purity(labels_, subtype), 2),
                     "inversions in the tree": analysis.count_inversions(Z_)}
pd.DataFrame(rows).T

,cluster sizes (k = 5),purity vs PAM50,inversions in the tree
ward,"[711, 528, 312, 212, 141]",0.49,0
centroid,"[1900, 1, 1, 1, 1]",0.36,403
average,"[1892, 8, 2, 1, 1]",0.36,0
complete,"[1727, 98, 60, 18, 1]",0.41,0
single,"[1900, 1, 1, 1, 1]",0.36,0


Only **Ward** gives balanced clusters. Single linkage "chains" everything into
one giant cluster and isolates single patients (the *chaining* of lecture 05);
average linkage is almost as extreme, and complete linkage still puts ~90% of
the patients in one cluster. **Centroid linkage** fails as well, and its tree
has hundreds of **inversions** (a merge drawn lower than the previous one),
the drawback lecture 06 warns about: such a tree cannot be cut cleanly at a
height. These failures are typical of high-dimensional data, where all
distances are similar (the *norm concentration* of the dimensionality
reduction lecture). Ward is therefore the default linkage; the others remain
available in the dashboard to show the effect.

The dendrogram (Ward, top 30 merges shown; the dashed line is the cut that
gives 5 clusters):

In [36]:
Z = analysis.linkage_tree(X, "ward")
dendro = hierarchy.dendrogram(Z, truncate_mode="lastp", p=30, no_plot=True)
fig = go.Figure()
for xs, ys in zip(dendro["icoord"], dendro["dcoord"]):
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=figures.TEXT_SECONDARY, width=1.5),
                             hoverinfo="skip", showlegend=False))
cut = (Z[-5, 2] + Z[-4, 2]) / 2                       # height between the 4th and 5th last merges
fig.add_hline(y=cut, line_dash="dash", line_color=figures.CATEGORICAL[1],
              annotation_text="cut → 5 clusters", annotation_position="top right")
leaf_x = 5 + 10 * np.arange(len(dendro["ivl"]))
fig.update_xaxes(tickvals=leaf_x, ticktext=dendro["ivl"], tickangle=-90, showgrid=False,
                 title="Patients (in brackets: number of patients in a collapsed branch)")
fig.update_layout(title="Ward dendrogram of the patients (truncated to the last 30 merges)",
                  yaxis_title="Merge distance (Ward)", height=440)
fig

**Who is in the small clusters?** Complete linkage isolates a group of a few
dozen patients or less; let us look at the smallest non-trivial one:

In [37]:
complete5 = analysis.hierarchical(X, 5, "complete")
sizes = np.bincount(complete5)
group_id = min((size, i) for i, size in enumerate(sizes) if size > 5)[1]
outliers = complete5 == group_id
print(f"Smallest non-trivial complete-linkage cluster: {outliers.sum()} patients")
print("  cohort :", cohort[outliers].value_counts().to_dict())
print("  subtype:", subtype[outliers].value_counts().to_dict())
gap = (mrna[outliers].mean() - mrna[~outliers].mean()).abs().nlargest(6)
print("  genes where they differ most (difference of mean raw z-score):",
      {g.upper(): round(v, 1) for g, v in gap.items()})

Smallest non-trivial complete-linkage cluster: 18 patients
  cohort : {'1': 18}
  subtype: {'LumB': 11, 'claudin-low': 4, 'Basal': 1, 'LumA': 1, 'Her2': 1}
  genes where they differ most (difference of mean raw z-score): {'SMAD5': 6.9, 'MSH2': 5.7, 'PPP2CB': 4.9, 'EPCAM': 4.4, 'TAF4B': 4.1, 'PIK3CA': 4.0}


It is a group of **18 patients, all from cohort 1**, with extreme values
(z ≈ +4 to +7) on a handful of unrelated genes, and of mixed subtypes. Coming
from a single batch, it is most likely a **technical artefact** rather than a
tumour type (it is also the small island seen in t-SNE and UMAP). The
dashboard must let the user spot and exclude such groups (select them in a
plot, then filter them out).

### 8.3 DBSCAN (density-based)

DBSCAN groups points that have at least `min_samples` neighbours within a
radius `eps`, and labels isolated points as noise. To choose `eps`, the
practical session's heuristic is the **sorted k-distance plot**: the distance
of each point to its k-th nearest neighbour; the "knee" of the curve is a
good `eps`.

In [38]:
kd_high = analysis.k_distance(X, 5)
kd_umap = analysis.k_distance(U, 10)
fig = make_subplots(rows=1, cols=2, subplot_titles=["489-D expression space (k = 5)", "UMAP 2-D space (k = 10)"])
fig.add_trace(go.Scatter(y=kd_high, mode="lines", line=dict(color=figures.CATEGORICAL[0], width=2), showlegend=False,
                         hovertemplate="%{y:.2f}<extra></extra>"), 1, 1)
fig.add_trace(go.Scatter(y=kd_umap, mode="lines", line=dict(color=figures.CATEGORICAL[0], width=2), showlegend=False,
                         hovertemplate="%{y:.2f}<extra></extra>"), 1, 2)
fig.update_xaxes(title_text="Patients sorted by distance")
fig.update_yaxes(title_text="Distance to k-th neighbour", row=1, col=1)
fig.update_annotations(font_size=12)
fig.update_layout(title="k-distance plots (DBSCAN eps heuristic)", height=340, margin=dict(t=70))
fig

In [39]:
print(f"489-D: the 5-NN distance of every patient lies between {kd_high.min():.1f} and {kd_high.max():.1f}, "
      f"median {np.median(kd_high):.1f}")
for eps in [18, 22, 26, 28]:
    lab = analysis.dbscan(X, eps=eps, min_samples=5)
    sizes_ = sorted(np.bincount(lab[lab >= 0]).tolist(), reverse=True) if (lab >= 0).any() else []
    print(f"  DBSCAN eps={eps}: {len(sizes_)} cluster(s) of sizes {sizes_}, {np.mean(lab == -1):.0%} noise")

489-D: the 5-NN distance of every patient lies between 18.1 and 48.6, median 23.3
  DBSCAN eps=18: 2 cluster(s) of sizes [5, 5], 99% noise
  DBSCAN eps=22: 2 cluster(s) of sizes [835, 6], 56% noise


  DBSCAN eps=26: 1 cluster(s) of sizes [1636], 14% noise
  DBSCAN eps=28: 1 cluster(s) of sizes [1789], 6% noise


In the 489-D space, the k-distance curve rises slowly for 97% of the patients
and bends only for the last few (outliers): all patients are roughly equally
far from their neighbours (the *curse of dimensionality*). An eps at the knee
(≈ 28) puts almost everybody in a single cluster; a smaller eps declares most
patients noise. No value of eps gives more than one substantial cluster (the
occasional second cluster holds 5–6 patients): **there are no
density-separated groups in the expression space.**

Density-based clustering becomes usable in a low-dimensional embedding. On the
UMAP coordinates, the result depends strongly on eps:

In [40]:
rows = []
for eps in [0.2, 0.25, 0.3, 0.4, 0.5]:
    lab = analysis.dbscan(U, eps=eps, min_samples=10)
    sizes_ = sorted(np.bincount(lab[lab >= 0]).tolist(), reverse=True)
    rows.append({"eps": eps, "clusters": len(sizes_), "noise": f"{np.mean(lab == -1):.0%}",
                 "largest clusters": sizes_[:6], "purity vs PAM50": round(analysis.purity(lab, subtype), 2),
                 "18-patient group is its own cluster": any(set(np.flatnonzero(lab == k)) == set(np.flatnonzero(outliers))
                                                            for k in set(lab) if k >= 0)})
pd.DataFrame(rows).set_index("eps")

,clusters,noise,largest clusters,purity vs PAM50,18-patient group is its own cluster
eps,,,,,
0.20,17,13%,"[1063, 177, 111, 72, 39, 30]",0.53,True
0.25,7,4%,"[1443, 232, 78, 43, 18, 10]",0.44,True
0.30,4,1%,"[1736, 91, 45, 18]",0.36,True
0.40,2,0%,"[1886, 18]",0.36,True
0.50,2,0%,"[1886, 18]",0.36,True


In [41]:
EPS = 0.25
db = analysis.dbscan(U, eps=EPS, min_samples=10)
db_labels = pd.Series(np.where(db == -1, "noise", [f"D{c + 1}" for c in db]), index=clinical.index)
db_order = sorted(set(db_labels) - {"noise"}, key=lambda s: int(s[1:])) + ["noise"]
db_colors = {**figures.color_map("dbscan", db_order[:-1]), "noise": figures.NOISE_COLOR}

fig = make_subplots(rows=1, cols=1)
embedding_scatter(fig, U, db_labels, db_colors, db_order)
fig.update_layout(title=f"DBSCAN clusters on the UMAP embedding (eps = {EPS}, min_samples = 10)",
                  height=440, legend_title_text="DBSCAN cluster")
fig

With a small eps (0.2), DBSCAN cuts the cloud into many clusters and calls 13%
of the patients noise; at the knee of the k-distance curve (≈ 0.3) and above,
it finds one large component plus small groups. In every setting the
18-patient artefact is isolated: here density-based clustering is mostly an
**outlier detector**, which is itself a useful insight for the user. Two
caveats: clustering a UMAP embedding inherits UMAP's distortions (section 7.5),
and the sensitivity to eps is exactly why the dashboard must expose eps with
the k-distance plot next to it.

**Summary of the clustering experiments.** The expression data is a continuum
rather than a set of separated groups. K-means and Ward recover the main
biological axis (ER+/luminal vs ER−/basal) and clusters with different
prognosis; single/average linkage and DBSCAN mostly isolate outliers. This
tells us what the dashboard must support: **comparing algorithms and
parameters, and judging a clustering against clinical labels and survival**,
rather than presenting one "true" clustering.

## 9. From the analysis to the dashboard

### 9.1 Questions a user will ask

| # | Question | Dashboard feature |
|---|---|---|
| Q1 | What does my (filtered) cohort look like? | Global filters (subtype, receptors, age, stage, treatment…) updating every view; summary tiles; distributions with mean/median markers |
| Q2 | How do the clinical variables relate to each other and to subtypes? | Spearman correlation and mutual-information matrices: click a cell to open that pair as a scatter / cross-table |
| Q3 | Do subtypes (or any group I select) have different survival? | survival over the first 5 years (share alive per month) split by any variable or by the current selection |
| Q4 | Which genes are expressed differently, and which are mutated, in a group of patients? | Select patients anywhere → mutation frequencies "selection vs rest"; mean-expression heatmap of subtypes / clusters with its dendrogram |
| Q5 | Is there structure in the gene expression? | PCA / t-SNE / UMAP view with adjustable parameters, R<sub>NX</sub> quality curve, colour by any variable or gene |
| Q6 | Do unsupervised clusters match known subtypes or prognosis? | K-means / hierarchical (dendrogram) / DBSCAN with parameters, choose-k plots, cluster × subtype table, survival per cluster |
| Q7 | Are there outliers or technical artefacts? | Lasso a group in the projection → see who they are (cohort, subtype) → exclude them |

### 9.2 Implications for the design

- **One colour per subtype everywhere**; clusters use a different palette so a
  cluster is never mistaken for a subtype.
- **Survival = share alive per month among patients with a known status** (first 5 years),
  never "% deceased".
- **Skewed variables** (nodes, size) need log/binned scales.
- **Batch effect** documented in this notebook only; the dashboard shows the
  expression data as published.
- **Heavy algorithms** (t-SNE, UMAP, hierarchical) run on demand with a
  "Run" button and are cached; PCA is precomputed.
- **Every selection is linked**: a lasso in a projection, a click on a cluster,
  a dendrogram branch or a heatmap cell selects patients for all views.